# RAG Evaluation — BAD VERSION

## Intentional Problems in this pipeline:
| Problem | Impact |
|---|---|
| Only 2 of 10 topic documents indexed | Retriever can't answer most questions → Context Relevance ≈ 0 |
| `similarity_top_k=1` | Too little context for LLM |
| Large chunk size (1024), zero overlap | Noisy, disconnected chunks |
| Basic Vector Index, no post-processing | No context expansion |

**Expected scores:** Context Relevance ≈ 0–0.2, Groundedness < 0.3, Answer Relevance < 0.4

In [1]:
import nest_asyncio
nest_asyncio.apply()

import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your_google_api_key_here":
    raise ValueError("Set GOOGLE_API_KEY in your .env file!")
print("API Key loaded.")

API Key loaded.


##  BAD CHOICE 1: Only 2 topic documents (out of 10 needed)

In [2]:
import wikipediaapi
from llama_index.core.schema import Document

wiki = wikipediaapi.Wikipedia('RAGDemo/1.0', 'en')

# BAD: Only 2 topics — test questions cover 10 different topics
bad_topics = [
    'Overfishing',           # only vaguely related
    'Coastal management',    # only partially related
]

bad_documents = []
for topic in bad_topics:
    page = wiki.page(topic)
    if page.exists():
        bad_documents.append(Document(
            text=page.text,
            metadata={'title': topic, 'source': 'wikipedia'}
        ))
        print(f"Loaded: {topic} ({len(page.text):,} chars)")

print(f"\nBAD: Only {len(bad_documents)} documents loaded — 8 test questions have NO relevant context!")

Loaded: Overfishing (33,216 chars)
Loaded: Coastal management (34,484 chars)

BAD: Only 2 documents loaded — 8 test questions have NO relevant context!


## BAD CHOICE 2: Poor chunk settings + low top_k

In [3]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings, VectorStoreIndex

llm = GoogleGenAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY, temperature=0.1)
embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=GOOGLE_API_KEY)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 1024   # BAD: huge chunks — noisy, unfocused
Settings.chunk_overlap = 0   # BAD: context cut off at arbitrary points

# BAD: Basic index, no sentence window
bad_index = VectorStoreIndex.from_documents(bad_documents, show_progress=True)

# BAD: top_k=1 — only 1 chunk retrieved, almost never enough
bad_engine = bad_index.as_query_engine(similarity_top_k=1)

print("BAD pipeline: chunk_size=1024, overlap=0, top_k=1, only 2 docs")

/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

BAD pipeline: chunk_size=1024, overlap=0, top_k=1, only 2 docs


## Configure TruLens

In [4]:
from trulens.core import TruSession, Metric
from trulens.apps.llamaindex import TruLlama
from trulens.providers.litellm import LiteLLM
from trulens.feedback.groundtruth import GroundTruthAgreement

session = TruSession()
session.reset_database()

os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY
gemini_provider = LiteLLM(model_engine="gemini/gemini-2.5-flash")

qa_df = pd.read_csv("ipcc_test_questions.csv")
qa_set = [{"query": item["Question"], "response": item["Answer"]} for _, item in qa_df.iterrows()]

m_answer_relevance = Metric(implementation=gemini_provider.relevance_with_cot_reasons, name="Answer Relevance").on_input_output()
m_context_relevance = Metric(implementation=gemini_provider.context_relevance_with_cot_reasons, name="Context Relevance").on_prompt().on_context(collect_list=True)
m_groundedness = Metric(implementation=gemini_provider.groundedness_measure_with_cot_reasons, name="Groundedness").on_context(collect_list=True).on_response()
m_groundtruth = Metric(implementation=GroundTruthAgreement(qa_set, provider=gemini_provider).agreement_measure, name="Answer Correctness").on_input_output()

metrics = [m_answer_relevance, m_context_relevance, m_groundedness, m_groundtruth]

bad_recorder = TruLlama(bad_engine, app_name="BAD RAG", feedbacks=metrics)
print(f"TruLens ready. Running {len(qa_set)} queries...")

Package jsonschema not present in requirements.


instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index_instrumentation.DispatcherSpanMixin'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class '

/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
WARNI [trulens.feedback.computer] feedback_name=Answer Correctness, record=71ed9941-68d5-48b1-894d-a3a22976602a, span_group=None had an error during computation:
'expected_response'
/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
WARNI [trulens.feedback.computer] feedback_name=Answer Correctness, record=8a9b493a-38fa-401d-8713-ffcc05bec390, span_group=None had an error during computation:
'expected_response'
/Users/ratneshsingh/Developer/trulens/trulens-eval-demo/venv/lib/python3.14/site-packages/trulens/feedback/llm_provider.py:2857: UserWarning: Failed to proc

## Run Evaluation

In [5]:
print(f"Running {len(qa_set)} queries through BAD RAG...")
with bad_recorder as recording:
    for i, q in enumerate(qa_set):
        print(f"  [{i+1}/{len(qa_set)}] {q['query'][:65]}")
        bad_engine.query(q['query'])
print("\nBAD RAG evaluation complete.")

Running 10 queries through BAD RAG...
  [1/10] What are the primary impacts of climate change on ocean and coast
  [2/10] How do marine heatwaves affect ocean ecosystems?
  [3/10] What role does the ocean play in global climate regulation?
  [4/10] How is climate change impacting marine biodiversity?
  [5/10] What are the major non-climate drivers affecting ocean and coasta
  [6/10] How does climate change influence the distribution of marine spec
  [7/10] What are the effects of ocean acidification on marine life?
  [8/10] How does climate change affect coastal communities and industries
  [9/10] What are some adaptation strategies for managing climate change i
  [10/10] What is the significance of the paleorecord in understanding clim

BAD RAG evaluation complete.


In [ ]:
records, feedback = session.get_records_and_feedback(app_ids=[])
print(f"Total records: {len(records)}\n")

metric_cols = [c for c in records.columns if c in ["Answer Relevance", "Context Relevance", "Groundedness", "Answer Correctness"]]
print("=== BAD RAG Scores ===")
for col in metric_cols:
    score = records[col].mean()
    status = " Poor" if score < 0.4 else " Mediocre" if score < 0.6 else " Good" if score < 0.8 else "🌟 Excellent"
    print(f"  {col:25s}: {score:.3f}  {status}")

records[metric_cols + ['app_name']].head(10)

Total records: 8

=== BAD RAG Scores ===
  Answer Relevance         : 0.333  🛑 Poor
  Context Relevance        : 0.333  🛑 Poor
  Groundedness             : 1.000  🌟 Excellent
  Answer Correctness       : nan  🌟 Excellent


,Answer Relevance,Context Relevance,Groundedness,Answer Correctness,app_name
0,0.666667,0.666667,1.0,NaN,BAD RAG
1,0.000000,0.000000,1.0,NaN,BAD RAG
2,NaN,NaN,NaN,NaN,BAD RAG
3,NaN,NaN,NaN,NaN,BAD RAG
4,NaN,NaN,NaN,NaN,BAD RAG
5,NaN,NaN,NaN,NaN,BAD RAG
6,NaN,NaN,NaN,NaN,BAD RAG
7,NaN,NaN,NaN,NaN,BAD RAG


In [7]:
session.run_dashboard()

Starting dashboard ...


/var/folders/kr/732g325n4j9czjh939zyww3h0000gn/T/ipykernel_97613/1555659051.py:1: DeprecationWarning: Method `run_dashboard` has been renamed or moved to `trulens.dashboard.run.run_dashboard`.

  session.run_dashboard()


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:63641 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>